In [ ]:
!pip install -q "transformers==4.43.4" "bitsandbytes==0.43.1" einops pysam biopython scikit-learn

In [ ]:
import os
os.makedirs("/kaggle/working/raw", exist_ok=True)

# -c = resume if interrupted; ClinVar's rolling GRCh38 VCF + its tabix index
!wget -c -q --show-progress \
  https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz \
  -O /kaggle/working/raw/clinvar.vcf.gz
!wget -c -q --show-progress \
  https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz.tbi \
  -O /kaggle/working/raw/clinvar.vcf.gz.tbi

In [ ]:
# 1) Size sanity — should be tens-to-a-couple-hundred MB. A few KB = error page.
!ls -lh /kaggle/working/raw/clinvar.vcf.gz*

# 2) Provenance — freeze this snapshot's identity for the writeup
print("\n--- release / fileDate ---")
!zcat /kaggle/working/raw/clinvar.vcf.gz | grep -m1 '^##fileDate'
!zcat /kaggle/working/raw/clinvar.vcf.gz | grep -m3 -i '^##source\|^##ID\|ClinVar' | head -3

# 3) Contig naming — MUST be 1,2,...,X (NCBI/Ensembl style, no "chr"),
#    so it matches the Ensembl FASTA we pull next
print("\n--- CHROM of first 5 real records ---")
!zcat /kaggle/working/raw/clinvar.vcf.gz | grep -v '^#' | head -5 | cut -f1

# 4) Confirm the INFO fields our Phase-2 filter depends on are defined
print("\n--- INFO fields present? ---")
!zcat /kaggle/working/raw/clinvar.vcf.gz | grep '^##INFO' \
    | grep -E 'ID=(CLNSIG|CLNREVSTAT|GENEINFO|CLNVC|MC)='

In [ ]:
import subprocess, gzip

# Decompress just the header (stops at first non-# line) — no pipe, no race
wanted = ["CLNSIG", "CLNREVSTAT", "GENEINFO", "CLNVC", "MC"]
found = {w: False for w in wanted}

with gzip.open("/kaggle/working/raw/clinvar.vcf.gz", "rt") as f:
    for line in f:
        if not line.startswith("#"):
            break                      # header done; stop before the 2M+ records
        if line.startswith("##INFO"):
            for w in wanted:
                if f"ID={w}," in line:
                    found[w] = True

for w in wanted:
    print(f"{w:12} {'FOUND' if found[w] else 'MISSING'}")

In [ ]:
# Ensembl 116 (June 2026). primary_assembly = chr 1-22,X,Y,MT minus alt/patch contigs.
# ~950 MB compressed. -c resumes a broken transfer.
PRIMARY = "https://ftp.ensembl.org/pub/release-116/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz"
FALLBACK = "https://ftp.ebi.ac.uk/pub/ensembl/release-116/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz"
OUT = "/kaggle/working/raw/GRCh38.primary_assembly.fa.gz"

import subprocess
rc = subprocess.run(["wget","-c","-q","--show-progress",PRIMARY,"-O",OUT]).returncode
if rc != 0:
    print("Primary host failed, trying EBI mirror…")
    subprocess.run(["wget","-c","-q","--show-progress",FALLBACK,"-O",OUT], check=True)
print("done")

In [ ]:
!gunzip -k /kaggle/working/raw/GRCh38.primary_assembly.fa.gz   # ~3 GB uncompressed
import pysam
pysam.faidx("/kaggle/working/raw/GRCh38.primary_assembly.fa")
print("indexed")

In [ ]:
fa = "/kaggle/working/raw/GRCh38.primary_assembly.fa"

!ls -lh /kaggle/working/raw/GRCh38.primary_assembly.fa*

print("\n--- contig names (should be 1,2,...,X,Y,MT + scaffolds) ---")
!cut -f1 /kaggle/working/raw/GRCh38.primary_assembly.fa.fai | head -30

import pysam, gzip
ref = pysam.FastaFile(fa)
with gzip.open("/kaggle/working/raw/clinvar.vcf.gz", "rt") as f:
    for line in f:
        if not line.startswith("#"):
            chrom, pos, _id, refallele = line.split("\t")[:4]
            break
pos = int(pos)
fetched = ref.fetch(chrom, pos - 1, pos - 1 + len(refallele)).upper()   # 1-based VCF → 0-based pysam
print(f"\nClinVar: CHROM={chrom} POS={pos} REF={refallele}")
print(f"FASTA  : {fetched}")
print("MATCH ✅" if fetched == refallele.upper() else "MISMATCH ❌ — stop, do not proceed")

In [ ]:
import pandas as pd, glob, shutil, os

cands = glob.glob("/kaggle/input/**/Census_all*.tsv", recursive=True)
print("found:", cands)
src = cands[0]

cgc = pd.read_csv(src, sep="\t")
print("\nshape:", cgc.shape)
print("columns:", list(cgc.columns))

# locate the columns we care about, robust to exact naming
geneid_col = [c for c in cgc.columns if "entrez" in c.lower()][0]
tier_col   = [c for c in cgc.columns if c.strip().lower() == "tier"][0]
print(f"\nGeneID column: '{geneid_col}' | Tier column: '{tier_col}'")

# GeneID populated?
cgc[geneid_col] = pd.to_numeric(cgc[geneid_col], errors="coerce").astype("Int64")
n_total, n_id = len(cgc), int(cgc[geneid_col].notna().sum())
print(f"\ngenes: {n_total} | with Entrez GeneId: {n_id} | missing: {n_total - n_id}")
print("\ntier breakdown:")
print(cgc[tier_col].value_counts(dropna=False))

# copy to clean, persistent name
os.makedirs("/kaggle/working/raw", exist_ok=True)
dst = "/kaggle/working/raw/cancer_gene_census.tsv"
shutil.copy(src, dst)
print("\ncopied ->", dst)

In [ ]:
import gzip, re

cgc_ids = set(int(x) for x in cgc[geneid_col].dropna().tolist())

# extract every GeneID from ClinVar's GENEINFO (format: SYMBOL:ID or SYM1:ID1|SYM2:ID2)
clinvar_ids = set()
pat = re.compile(r"GENEINFO=([^;]+)")
with gzip.open("/kaggle/working/raw/clinvar.vcf.gz", "rt") as f:
    for line in f:
        if line[0] == "#":
            continue
        m = pat.search(line)
        if m:
            for tok in m.group(1).split("|"):
                p = tok.split(":")
                if len(p) >= 2 and p[-1].isdigit():
                    clinvar_ids.add(int(p[-1]))

overlap = cgc_ids & clinvar_ids
print(f"CGC genes (with GeneID)          : {len(cgc_ids)}")
print(f"distinct GeneIDs in ClinVar      : {len(clinvar_ids)}")
print(f"CGC genes present in ClinVar     : {len(overlap)}  ({100*len(overlap)/len(cgc_ids):.1f}%)")
missing = sorted(cgc_ids - clinvar_ids)
print(f"CGC genes with NO ClinVar variant: {len(missing)}  e.g. {missing[:10]}")

In [ ]:
import os
raw = "/kaggle/working/raw"
for f in sorted(os.listdir(raw)):
    p = os.path.join(raw, f)
    print(f"{os.path.getsize(p)/1e6:8.1f} MB   {f}")

In [ ]:
import os
os.remove("/kaggle/working/raw/GRCh38.primary_assembly.fa.gz")
print("removed .fa.gz; remaining:")
for f in sorted(os.listdir("/kaggle/working/raw")):
    print(" ", f)

In [ ]:
import json, os

raw = "/kaggle/working/raw"
# metadata: "id" must be <your-kaggle-username>/<slug>
meta = {
    "title": "cancer-variant-lm-raw",
    "id": "lazarvelinov46/cancer-variant-lm-raw",
    "licenses": [{"name": "other"}]   # mixed sources incl. COSMIC (restricted) → not a permissive license
}
with open(os.path.join(raw, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

# --dir-mode zip packs the big .fa efficiently for upload
!kaggle datasets create -p {raw} --dir-mode zip